<a href="https://colab.research.google.com/github/syedmahmoodiagents/GraphRAG/blob/main/GraphRAG_Simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
# !pip install -q langgraph langchain-huggingface --q

In [39]:
import os, getpass

In [40]:
os.environ["HF_TOKEN"] = getpass.getpass()

In [41]:

from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint

In [42]:
llm = ChatHuggingFace(llm=HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b"))

In [43]:
doc = {
    "Transformer": """
The Transformer is a neural network architecture introduced in 2017.
It relies entirely on attention mechanisms.
""",
    "Attention": """
Attention allows the model to focus on relevant parts of the input sequence.
Self-attention computes relationships within the same sequence.
""",
    "Encoder": """
The encoder maps input tokens into contextual embeddings.
""",
    "Decoder": """
The decoder generates output tokens one step at a time.
"""
}


In [44]:
graph_db = {}

def ingest_document(doc: dict):
    for section, text in doc.items():
        graph_db[section] = {
            "content": text.strip(),
            "neighbors": []
        }

    # add relationships (could be LLM-driven)
    graph_db["Transformer"]["neighbors"] = ["Attention", "Encoder", "Decoder"]
    graph_db["Attention"]["neighbors"] = ["Self-Attention"]


In [45]:

ingest_document(doc)

In [46]:
graph_db

{'Transformer': {'content': 'The Transformer is a neural network architecture introduced in 2017.\nIt relies entirely on attention mechanisms.',
  'neighbors': ['Attention', 'Encoder', 'Decoder']},
 'Attention': {'content': 'Attention allows the model to focus on relevant parts of the input sequence.\nSelf-attention computes relationships within the same sequence.',
  'neighbors': ['Self-Attention']},
 'Encoder': {'content': 'The encoder maps input tokens into contextual embeddings.',
  'neighbors': []},
 'Decoder': {'content': 'The decoder generates output tokens one step at a time.',
  'neighbors': []}}

In [47]:
class GraphRAGState(TypedDict):
    query: str
    entities: List[str]
    matched_nodes: List[Dict]
    graph_context: str
    answer: str

In [48]:
def extract_entities(state: GraphRAGState):
    query = state["query"]

    # simple heuristic entity matcher
    entities = [
        node for node in graph_db.keys()
        if node.lower() in query.lower()
    ]

    return {"entities": entities}

In [49]:
def traverse_graph(state: GraphRAGState):
    matched = []

    for entity in state["entities"]:
        node = graph_db.get(entity)
        if not node:
            continue

        matched.append({
            "entity": entity,
            "content": node["content"],
            "neighbors": node["neighbors"]
        })

        # 1-hop traversal
        for neighbor in node["neighbors"]:
            if neighbor in graph_db:
                matched.append({
                    "entity": neighbor,
                    "content": graph_db[neighbor]["content"],
                    "neighbors": graph_db[neighbor]["neighbors"]
                })

    return {"matched_nodes": matched}

In [50]:

def build_context(state: GraphRAGState):
    context = ""

    for node in state["matched_nodes"]:
        context += f"""
Entity: {node['entity']}
Description: {node['content']}
Related: {', '.join(node['neighbors'])}
"""

    return {"graph_context": context}


In [51]:
def generate_answer(state: GraphRAGState):
    prompt = f"""
You must answer ONLY using the context below.
If the answer is not in the context, say "I don't know".

Context:
{state['graph_context']}

Question:
{state['query']}
"""

    response = llm.invoke(prompt)
    return {"answer": response.content}

In [52]:

builder = StateGraph(GraphRAGState)

builder.add_node("extract", extract_entities)
builder.add_node("traverse", traverse_graph)
builder.add_node("context", build_context)
builder.add_node("generate", generate_answer)

builder.set_entry_point("extract")
builder.add_edge("extract", "traverse")
builder.add_edge("traverse", "context")
builder.add_edge("context", "generate")
builder.add_edge("generate", END)

graph = builder.compile()

In [53]:

result = graph.invoke({
    "query": "Explain how Transformer attention works"
})



In [54]:
print(result["answer"])

Transformer attention works by using attention mechanisms to allow the model to focus on relevant parts of the input sequence. In a Transformer, self‑attention is used, which computes relationships within the same sequence, enabling each token to consider every other token when forming its contextual embedding.
